# 06 — Judge Robustness

Reproduces paper **Tables 4 & 5** (cross-judge agreement on per-configuration
and per-concept best-utility, computed across the matched 88,000-response set).

Reads the `judge_gemma3_12b/`, `judge_gpt4_1_mini/`, and `judge_nova_2_lite/`
summary CSVs under `results/steering_evaluations/`.

> **Note:** Only the `judge_gemma3_12b` evaluations are included in the
> repository. To reproduce cross-judge comparisons (Tables 4 & 5), run the
> evaluation pipeline with the `judge_gpt4_1_mini` and `judge_nova_2_lite`
> judge tags first (see README).

In [11]:
import os
from pathlib import Path
import pandas as pd
from scipy.stats import pearsonr, spearmanr

# Ensure working directory is the repo root so relative paths resolve correctly.
_repo_root = Path(__file__).resolve().parent.parent if '__file__' in dir() else Path.cwd()
if (_repo_root / 'grace').is_dir():
    os.chdir(_repo_root)
elif (_repo_root.parent / 'grace').is_dir():
    os.chdir(_repo_root.parent)

from grace.analysis.load_results import load_summary_results

MODEL = 'google/gemma-3-27b-it'
CONCEPTS = sorted(p.stem for p in Path('concepts/gpt-5/extract').glob('*.json'))
JUDGES = {'gemma3-12b': 'judge_gemma3_12b', 'gpt-4.1-mini': 'judge_gpt4_1_mini', 'nova-2-lite': 'judge_nova_2_lite'}

print(f"CWD: {Path.cwd()}")
print(f"Concepts: {len(CONCEPTS)}")

CWD: /home/jtr2875/GRACE
Concepts: 20


In [12]:
# Fixed interval sweep grid (same as notebook 01)
FIXED_LAYERS = [5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55, 60]
FIXED_COEFS = [1.0, 2.0, 3.0]

all_rows = []
for jname, jtag in JUDGES.items():
    for c in CONCEPTS:
        for r in load_summary_results(MODEL, c, judge_tag=jtag, method='pv'):
            if r['layer'] in FIXED_LAYERS and r['coef'] in FIXED_COEFS:
                r['judge'] = jname
                all_rows.append(r)
df = pd.DataFrame(all_rows)
print(f"{len(df)} rows (fixed sweep only: {len(FIXED_LAYERS)} layers x {len(FIXED_COEFS)} coefs)")
df.head()

2160 rows (fixed sweep only: 12 layers x 3 coefs)


,concept,method,layer,coef,mean_concept_score,mean_coherence,mean_utility,judge
0,anthropomorphic,pv,50,2.0,86.97,85.28,86.125,gemma3-12b
1,anthropomorphic,pv,20,2.0,75.06,90.96,83.010,gemma3-12b
2,anthropomorphic,pv,35,2.0,91.95,87.72,89.835,gemma3-12b
3,anthropomorphic,pv,60,2.0,87.14,71.04,79.090,gemma3-12b
4,anthropomorphic,pv,40,3.0,95.55,19.99,57.770,gemma3-12b


## Table 4 — Per-configuration agreement (concept score and coherence)


In [13]:
pivot_cs = df.pivot_table(index=['concept', 'layer', 'coef'], columns='judge', values='mean_concept_score')
pivot_co = df.pivot_table(index=['concept', 'layer', 'coef'], columns='judge', values='mean_coherence')
judges = list(JUDGES)
table4 = []
for i in range(len(judges)):
    for j in range(i + 1, len(judges)):
        a, b = judges[i], judges[j]
        if a in pivot_cs and b in pivot_cs:
            pair_cs = pivot_cs[[a, b]].dropna()
            pair_co = pivot_co[[a, b]].dropna()
            if len(pair_cs) < 3 or len(pair_co) < 3:
                continue
            cs_p, _ = pearsonr(pair_cs[a], pair_cs[b])
            cs_s, _ = spearmanr(pair_cs[a], pair_cs[b])
            co_p, _ = pearsonr(pair_co[a], pair_co[b])
            co_s, _ = spearmanr(pair_co[a], pair_co[b])
            table4.append({'pair': f'{a} vs {b}', 'cs_r': cs_p, 'cs_rho': cs_s,
                           'co_r': co_p, 'co_rho': co_s, 'n': len(pair_cs)})
pd.DataFrame(table4).round(2)

,pair,cs_r,cs_rho,co_r,co_rho,n
0,gemma3-12b vs gpt-4.1-mini,0.88,0.91,0.93,0.52,720
1,gemma3-12b vs nova-2-lite,0.82,0.88,0.97,0.74,720
2,gpt-4.1-mini vs nova-2-lite,0.94,0.96,0.97,0.76,720


## Table 5 — Per-concept best-utility agreement


In [14]:
best = df.groupby(['judge', 'concept'])['mean_utility'].max().reset_index()
wide = best.pivot(index='concept', columns='judge', values='mean_utility')
table5 = []
for i in range(len(judges)):
    for j in range(i + 1, len(judges)):
        a, b = judges[i], judges[j]
        if a in wide and b in wide:
            pair = wide[[a, b]].dropna()
            if len(pair) < 3:
                continue
            r, _ = pearsonr(pair[a], pair[b])
            rho, _ = spearmanr(pair[a], pair[b])
            table5.append({'pair': f'{a} vs {b}', 'pearson_r': r, 'spearman_rho': rho, 'n': len(pair)})
pd.DataFrame(table5).round(2)

,pair,pearson_r,spearman_rho,n
0,gemma3-12b vs gpt-4.1-mini,0.94,0.72,20
1,gemma3-12b vs nova-2-lite,0.90,0.68,20
2,gpt-4.1-mini vs nova-2-lite,0.92,0.94,20
